In [10]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("./src")

import numpy as np
import pandas as pd
from datetime import datetime as dt
import matplotlib.pyplot as plt

from data_loader import (
    load_secrets, get_tiingo_client, load_all_assets, clean_data,
    compute_volatility, load_tbill_data, load_equity_market_data,
    get_option_inputs,
)
from pricing import black_scholes, delta, gamma, theta, vega, rho, build_greeks_table, build_bs_price_table
from monte_carlo import simulate_terminal, build_mc_price_table
from implied_vol_BS import iv_NR, iv_brentq, iv_solve, build_iv_table
from option_api import fetch_option_chain

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
DATA_DIR = "./data"

secrets = load_secrets(secrets_path="./secrets.json")
client = get_tiingo_client(secrets)

dataframes = load_all_assets(data_dir=DATA_DIR)
df_tbill = load_tbill_data(data_dir=DATA_DIR)

dataframes = {key: clean_data(df) for key, df in dataframes.items()}
dataframes = {key: compute_volatility(df) for key, df in dataframes.items()}

print("Bond (Long) - ZFL.TO:", dataframes["bond1"].head())
print("Bond (Mid) - ZFM.TO:", dataframes["bond2"].head())
print("Equity - VFV.TO:", dataframes["equity"].head())
print("Gold - CGL.TO:", dataframes["gold"].head())

In [ ]:
df_tbill = load_tbill_data()
df_tbill.tail()

In [ ]:
symbol = "NVDA"
df_nvda = load_equity_market_data(symbol, client)

optexp = dt(2026, 7, 31)   # expiration date matches the option chain queried below

inputs = get_option_inputs(df_nvda, df_tbill, optexp)
S = inputs["S"]
T = inputs["T"]
r = inputs["r"]
sigma = inputs["sigma"]
valuation_date = inputs["valuation_date"]
log_returns = inputs["log_returns"]

print("Spot price (S):", S)
print("Time to expiration (T, years):", T)
print("Risk-free rate (r):", r)
print("Historical volatility (sigma):", sigma)
print("Valuation date:", valuation_date)

In [ ]:
opt_chain, resolved_date = fetch_option_chain(
    symbol,
    optexp.strftime("%Y-%m-%d"),
    valuation_date,
    S,
    option_type="call",
    strike_pct_low=0.91,
    strike_pct_high=1.16,
)

print("Resolved valuation date:", resolved_date)
opt_chain[["strike", "bid", "ask", "Pmkt"]]

In [ ]:
K = opt_chain["strike"].tolist()
print("Strikes in range:", K)

In [ ]:
bs_prices = build_bs_price_table(S, K, T, r, sigma)
bs_prices

In [ ]:
greeks_call = build_greeks_table(S, K, T, r, sigma, option_type="call")
greeks_put = build_greeks_table(S, K, T, r, sigma, option_type="put")

print(f"Greeks for {symbol} call")
display(greeks_call)

print(f"Greeks for {symbol} put")
display(greeks_put)

In [ ]:
mc_prices = build_mc_price_table(S, r, sigma, T, K, n_sims=10_000_000, seed=42)
mc_prices

In [ ]:
bs_iv = build_iv_table(S, opt_chain, T, r, option_type="call")
bs_iv

In [ ]:
strike_test = opt_chain["strike"].iloc[0]
pmkt_test = opt_chain["Pmkt"].iloc[0]

iv_test = iv_solve(S, strike_test, T, r, pmkt_test)
recovered_price = black_scholes(S, strike_test, T, r, iv_test, option_type="call")

print(f"Strike: {strike_test:.2f}")
print(f"Implied Vol: {iv_test:.3f}")
print(f"Historical sigma: {sigma:.3f}")
print(f"Recovered Price: {recovered_price:.4f} vs Pmkt: {pmkt_test:.4f}")

In [ ]:
comparison = bs_prices.merge(mc_prices, on="Strike Price").merge(
    opt_chain[["strike", "Pmkt"]].rename(columns={"strike": "Strike Price"}),
    on="Strike Price",
    how="left",
)
comparison

In [ ]:
import plotly.express as px

fig = px.line(
    bs_iv.dropna(subset=["IV"]),
    x="Strike Price",
    y="IV",
    markers=True,
    title=f"{symbol} Implied Volatility Smile — Expiration {optexp.strftime('%Y-%m-%d')}",
)
fig.update_layout(yaxis_tickformat=".1%")
fig.show()

In [ ]:
# Plotting historical prices for all assets
plt.figure(figsize=(12,6))
for key, df in dataframes.items():
    plt.plot(df.index, df['Close'], label=key)
plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Historical Price Data')
plt.legend()
plt.show()

In [ ]:
# Plotting volatility
plt.figure(figsize=(12,6))
for key, df in dataframes.items():
    plt.plot(df.index, df['Volatility'], label=key)
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.title('Historical Volatility Data')
plt.legend()
plt.show()

In [ ]:
# Monte Carlo convergence: pricing error vs number of simulations
Ns = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]

# Study one strike (ATM = closest to spot)
strike = min(K, key=lambda k: abs(k - S))
bs = black_scholes(S, strike, T, r, sigma, 'call')   # exact reference value

errors = []
for n in Ns:
    St_n = simulate_terminal(S, r, sigma, T, n)
    mc = np.exp(-r*T) * np.maximum(St_n - strike, 0).mean()
    errors.append(abs(mc - bs))

plt.figure(figsize=(10, 6))
plt.loglog(Ns, errors, 'o-', label="|MC - B-S|")
# Theoretical 1/sqrt(N) reference, anchored at the first point
plt.loglog(Ns, errors[0]*np.sqrt(Ns[0]/np.array(Ns)), 'k--', label=r"$\propto 1/\sqrt{N}$")
plt.xlabel("Number of simulations (N)")
plt.ylabel("Absolute pricing error")
plt.title(f"{symbol}: Monte Carlo convergence (strike={strike:.0f})")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.show()